In [1]:
import sys
sys.path.insert(0, '..')
from support.paths import resolve

import pandas as pd
import os

In [2]:
%run ../0_Config/0_variables.ipynb

In [ ]:
def train_valid_test_split(feature_path: str, targets_path: str, selected_path: str):

    features = pd.read_parquet(feature_path)
    targets = pd.read_parquet(targets_path)
    selected = pd.read_parquet(selected_path)

    # TODO replace with selected features for each 30 min horizon (atm same full features for every horizon) and feature are at 5 min intervals
    selected_features = features 

    # Filter features to start at TRAIN_START so training begins on the intended date.

    cutoff_train = pd.Timestamp(os.environ["TRAIN_START"])
    cutoff_test = pd.Timestamp(os.environ["TEST_START"])
    cutoff_valid = pd.Timestamp(os.environ["VALID_START"])

    X_train = selected_features[(selected_features.index >= cutoff_train) & (selected_features.index <= cutoff_valid)]
    X_validate = selected_features[(selected_features.index > cutoff_valid) & (selected_features.index <= cutoff_test)]
    X_test  = selected_features[selected_features.index > cutoff_test]

    y_train = targets[targets.index <= cutoff_valid]
    y_validate = targets[(targets.index > cutoff_valid) & (targets.index <= cutoff_test)]
    y_test  = targets[targets.index > cutoff_test]

    X_train.to_parquet("Data/1_train_test_splits/X_train.parquet")
    X_validate .to_parquet("Data/1_train_test_splits/X_validate.parquet")
    X_test .to_parquet("Data/1_train_test_splits/X_test.parquet")
    y_train .to_parquet("Data/1_train_test_splits/y_train.parquet")
    y_validate .to_parquet("Data/1_train_test_splits/y_validate.parquet")
    y_test .to_parquet("Data/1_train_test_splits/y_test.parquet")

    display(X_train.shape)
    display(y_train.shape)

train_valid_test_split(
    feature_path = resolve(f"2_Features_build/Feature_data/{os.environ["FEATURE_DATASET"]}.parquet"),
    targets_path = resolve(f"3_Features_select/Selected_features/{os.environ['TARGET']}_targets_agg_{os.environ["FEATURE_DATASET"]}.parquet"),
    selected_path = resolve(f"3_Features_select/Selected_features/{os.environ['TARGET']}_selected_features_{os.environ["FEATURE_DATASET"]}.parquet")
)


(578305, 638)

(578305, 96)

In [4]:
%reset -f